# Matching donor draws

Matching transfers observed target values from donor records to recipients. Install the `matching` extra and R's `StatMatch` package before running this notebook. `predict` returns a DataFrame. Explicit quantiles and `return_probs=True` are unsupported. See the [migration guide](../../imputation-benchmarking/migration.md) for distributional model selection and compatibility.


In [ ]:
import numpy as np
import pandas as pd
from microimpute.models import Matching

rng = np.random.default_rng(42)
donor = pd.DataFrame(
    {"age": np.arange(20.0, 60.0), "income": rng.uniform(20_000, 60_000, 40)}
)
receiver = pd.DataFrame({"age": [25.0, 35.0, 45.0, 55.0]})

## Fit and draw

Fitting stores the donor data. Each prediction matches recipients to donors using the selected predictors and returns the donated target columns.


In [ ]:
fitted = Matching(seed=42).fit(donor, predictors=["age"], imputed_variables=["income"])
imputed_values = fitted.predict(receiver)
imputed_values.head()

## Weighted matching

`weight_col` supplies positive finite donor weights to R's `RANDwNND.hotdeck`. It retains the nearest-distance donor candidates and weights their selection. Weighted constrained matching is unsupported. Repeated calls advance the model's seed stream, though uniquely nearest donors can produce identical draws.


In [ ]:
weighted_donor = donor.assign(weight=np.linspace(1.0, 3.0, len(donor)))
weighted = Matching(seed=42).fit(
    weighted_donor, ["age"], ["income"], weight_col="weight"
)
weighted.predict(receiver)

## Evaluate the intended output

A donor draw is not a conditional quantile or a class probability forecast. Distributional cross-validation and predictor analysis reject Matching. For point-draw assessment, split donors and recipients and report a point-error metric explicitly. This example computes mean absolute error against held-out observed income; it does not estimate distributional calibration.


In [ ]:
training = donor.iloc[:30]
held_out = donor.iloc[30:]
fitted_holdout = Matching(seed=42).fit(training, ["age"], ["income"])
predicted = fitted_holdout.predict(held_out[["age"]])
mean_absolute_error = np.mean(
    np.abs(predicted.income.to_numpy() - held_out.income.to_numpy())
)
mean_absolute_error

To include Matching draws alongside distributional models, pass `models=[OLS, Matching]` and `impute_all=True` to `autoimpute`. Only models with quantile/probability forecasts enter the distributional ranking. Matching preserves donated values and within-record target combinations; nearest-neighbor selection does not guarantee that the recipient population reproduces the donor marginal distribution.
